In [1]:
import re
import csv
import time
import requests

import nltk
from bs4 import BeautifulSoup
from nltk.probability import FreqDist
import pythainlp


from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.pipeline import make_pipeline


In [ ]:



URL = 'https://www.siamzone.com/music/thailyric/'
HEADERS = {'User-Agent': 'KMITL NLP Student Bot'}

data = [
    ["Title", "Author", "Date", "Lyrics"]
]


for i in range (36575, 37470):
    try:
        print("Searching:", URL + str(i))

        response = requests.get(URL + str(i), headers=HEADERS)
        print(f"Request Status Code: {response.status_code}")

        soup = BeautifulSoup(response.content, 'html.parser')

        auth_date_elem = soup.select_one("div.has-text-grey:nth-child(5)")
        auth_date = auth_date_elem.get_text(strip=True).split("|")
        author = auth_date[0].strip().lstrip("ศิลปิน").strip()
        author = re.sub(r"\s+", " ", author)
        date = auth_date[1].strip().lstrip("เพิ่มเมื่อ").strip()

        title_elem = soup.select_one("div.column:nth-child(2) > div:nth-child(1) > h2:nth-child(1)")
        title = title_elem.get_text(strip=True).lstrip("เนื้อเพลง").strip()

        lyrics_elem = soup.select_one(".has-text-centered-mobile > div:nth-child(2)")
        lyrics = lyrics_elem.get_text(separator="\n", strip=True).strip()

        data.append([title, author, date, lyrics])
        print("Data retrieved.")

        time.sleep(0.2)
    
    except:
        print("URL not found: skip retrieval.")
        pass


Searching: https://www.siamzone.com/music/thailyric/36575
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36576
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36577
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36578
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36579
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36580
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36581
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36582
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36583
Request Status Code: 200
Data retrieved.
Searching: https://www.siamzone.com/music/thailyric/36584
Request Status Code: 200
Data retrieved.
Searching:

In [2]:
with open("output_file_2.csv", mode='w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(data)

print("Success")


Success


### Load ไฟล์ CSV จากการทำ Web Scraping

In [3]:
import csv
import pythainlp
import unicodedata
import nltk
from nltk.probability import FreqDist

corpus = []

# ถ้าใช้ output_file.csv ----
with open("output_file_2.csv", mode='r', newline='', encoding='utf-8') as csv_file:
    is_header = True
    reader = csv.reader(csv_file, delimiter=',')
    for row in reader:
        if is_header:
            is_header = False
            continue
        corpus.append(row[3]) # ---- เปลี่ยนเป็น row[2]


### ตัวอย่างเนื้อเพลงที่ได้

ใช้ตัวแปร `sample_corpus` เพื่อเก็บเนื้อเพลง 3 เพลงแรกและนำไปใช้เปรียบเทียบในขั้นถัดไป สังเกตว่า มีบางเพลงที่มีคำที่ไม่ใช่ภาษาไทยปะปนอยู่ร่วมด้วย

In [4]:
sample_corpus = corpus[:3]
print(sample_corpus)


["ยังมีเรื่องที่อยากทำอยู่เต็มกระดาน\r\nเริ่มจากทำเพลงแล้วให้เพลงมันทำงาน\r\nแล้วปัญหาของมึงก็เรื่องของมึงดิ\r\nถ้าเกาะผมมันไม่ดังคุณก็ทำกันเองดิ\r\nGet out บอกให้พวกมึงเดินหลีก\r\nเลือกจะเดินตรงแล้วปล่อยให้พวกมึงเดินดีด\r\nจุดไฟนำทางก็เพียงแค่เผามัน\r\nOh everyday hit the ควัน\r\nGet hype up with my beat\r\nดีดผมดีดเช้าดีดเย็น\r\nแค่เล็งให้ลงหลุมเรื่องลูกแก้วผมดีดเป็น\r\nมีความคิดจริตเย็น\r\nFor the next step ให้กับชีวิตต้องสิบเต็ม\r\nไม่รู้จะพูดยังไง Just wanna kiss her\r\nขาดเธอไม่ได้ก็ใจมัน Need ya\r\nAnd I'd love to know what's going on in your mind\r\nYou and me We decied\r\nWhen I wasn't drunk Crazy or high\r\nแล้วมันต้องมีสักวันและก่อนที่มันจะสาย\r\nจงเชื่อในสิ่งที่ทำให้มันมีความหมาย\r\nเหนื่อยไหมคนดี\r\nที่เธอมีพี่เป็นแฟน\r\nแก้วแหวนผมเคยมีแต่เอาไปจำนำทำนมให้เธอแทน\r\nพอมีคนที่รักพี่ทุ่มให้ทั้งใจ\r\nแล้วเธอกลายเป็นคนที่สวยแต่รูปไปได้ไง\r\nความสวยงามมันต้องงามจากข้างใน\r\nแบบเธอที่ฉันมองข้ามไปเพิ่งจะรู้ตัวและเข้าใจ\r\nชอบเธอนั้นมันก็คือเรื่องนึง\r\nแต่ถ้าเธอไปชอบเขานั่นมันก็เรื

### ทำความสะอาดข้อความ

ข้อมูลที่ได้จากการ scrape ถือว่าค่อนข้างสะอาดและมีรูปแบบที่แน่นอนแล้ว จากการตรวจสอบ พบจุดที่ต้องสนใจเป็นพิเศษ ดังนี้

1. เนื้อร้องมีวรรคแทรก ที่ขับขนานหรือเสริมกับคำร้องหลัก (backing vocals / bridge / ad-lib) ที่มักจะปิดด้วยวงเล็บเล็ก (`()`)
2. พบการใช้วงเล็บก้ามปู (`[]`) เช่น ในเพลงที่มีนักร้องหลายคน จะใช้เพื่อกำหนดผู้ร้องของวรรคนั้น ๆ ในเพลง "เริ่ด (Slay) - เฟลิซ (Felizz)" และ ใช้เพื่อบอกโครงสร้าง (`[Pre-hook]`) ในเพลง "พบเพื่อผ่าน - บิ๊ก ภูวดิท ศิลาอุดมเดช (OG-Anic)" 
3. การใช้เครื่องหมายจุดในบริบทต่าง ๆ เช่น `มาลี มาลี เด็กหญิงมาลี จบ ป.5 ก็ลาบุรีรัมย์` หรือ `ฝูงนกบินกลับฮัง แต่เฮากะยังสู้ต่อ.. โอ้ละหนอ ปานนี้อีพ่อสิเป็นจั่งได๋`
4. การทำ normalization กับตัวอักษรที่มี accent เช่น pinyin ในเพลง "Sky Lantern Wish (天灯愿) - บีเอ็นเค 48 (BNK48)" หรือ "Wang Pi Jao (หวังปี้เจ้า) - จาร์วิส (Jarvis)"
5. การทำ normalization กับตัวอักษรที่มี unicode ไม่สม่ำเสมอกับเพลงอื่น ๆ เช่น ในเพลง "ตัวปัญหา (Envy) - ทริปเปิล วี (VVV)" (ใช้ `′` แทน `'`) และเพลง "Beauty In You (Ost. ดีว่า..ราวี) - แจ็คกี้ ชาเคอลีน มึ้นช์" (ใช้ `е` (U+0435) แทน `e` (U+0065))
6. การจัดการคำสบถที่ต่างกัน เช่น คำที่ถูก sensor ในเนื้อเพลง "จับกุม - เวฟ (Wav)" กลับไม่ถูก sensor ในเพลง "ผู้พัน Remix - ยังเจ (Young J)" ทั้งนี้ พบว่าใช้ asterisk `*` ในการ sensor เสมอ
7. การสะกดคำ เช่น คำทับศัพท์ ชื่อเฉพาะ และอื่น ๆ


ดังนั้น จะกำหนดกระบวนการทำความสะอาดข้อความ ดังนี้
- Normalize ตัวอักษร ด้วยโมดูล  unicodedata (4, 5)
- ลบวงเล็บก้ามปู พร้อมข้อความด้านใน (2)
- ลบเครื่องหมายวรรคตอน ยกเว้น `*` `'` `(` และ `)`
- Normalize ข้อความด้วย [pythainlp.util.normalize](https://pythainlp.org/dev-docs/api/util.html#:~:text=pythainlp.util.normalize)
- แปลงอักษรละตินเป็น lower case


In [5]:
def clean_text(text: str):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("′", "'")
    text = re.sub(r"[^ก-๙\w\s\(\)\*']", "", text)

    text = pythainlp.util.normalize(text)
    return text.lower().strip()


### เปรียบเทียบระหว่างการทำ word-level และ subword-level tokenizer

ทดลองทำ tokenize กับ `sample_corpus` ด้วย pythainlp.tokenize.word_tokenize และ pythainlp.tokenize.subword_tokenize เพื่อแทนผลลัพธ์ของ word-level และ subword-level tokenization ตามลำดับ

In [6]:
# Word-level

tokens_word = []
for doc in sample_corpus:
    tok = pythainlp.tokenize.word_tokenize(clean_text(doc))
    tok = [t.strip() for t in tok if t.strip()]
    tokens_word.extend(tok)

dist_word = FreqDist(tokens_word)
print(dist_word)
print(dist_word.most_common(20))


<FreqDist with 402 samples and 1062 outcomes>
[('เธอ', 43), ('สอง', 29), ('ก็', 27), ('มี', 23), ('ให้', 21), ('แปด', 20), ('ไป', 18), ('มัน', 17), ('ที่', 13), ('สี่', 12), ('สาม', 12), ('จะ', 11), ('พี่', 11), ('มาลี', 11), ('ละ', 11), ('แล้ว', 10), ('ไม่', 10), ('ทำ', 9), ('ผม', 9), ('ได้', 9)]


In [7]:
# Subword-level

tokens_subw = []
for doc in sample_corpus:
    tok = pythainlp.tokenize.subword_tokenize(clean_text(doc))
    tok = [t.strip() for t in tok if t.strip()]
    tokens_subw.extend(tok)

dist_subw = FreqDist(tokens_subw)
print(dist_subw)
print(dist_subw.most_common(20))


<FreqDist with 290 samples and 2301 outcomes>
[('อ', 160), ('ง', 104), ('น', 102), ('ก', 71), ('ม', 65), ('ว', 52), ('ย', 48), ('ด', 47), ('e', 43), ('เธ', 43), ('ส', 43), ('บ', 42), ('ห', 40), ('a', 33), ('มี', 32), ('n', 31), ('มา', 30), ('t', 29), ('ก็', 28), ('i', 28)]


พบว่า แม้สังเกตจากจำนวนข้อมูล sample ที่ใช้ tokenize น้อย การใช้ word-level ได้ผล token ที่แตกต่างกัน (unique) ได้มากกว่า subword-level อย่างเห็นได้ชัด นั่นคือ word-level ได้ vocabulary size ที่มากกว่า
  
เปรียบเทียบการ tokenize ในเพลงหนึ่งเพลง

In [8]:
lyrics = clean_text(sample_corpus[2])

# Word-level
tokens_word_lyrics = pythainlp.tokenize.word_tokenize(lyrics)
tokens_word_lyrics = [t.strip() for t in tokens_word_lyrics if t.strip()]

# Subword-level
tokens_subw_lyrics = pythainlp.tokenize.subword_tokenize(lyrics)
tokens_subw_lyrics = [t.strip() for t in tokens_subw_lyrics if t.strip()]

print("Lyrics:")
print(lyrics)
print("\nWord-level Tokenizer:")
print(" | ".join(tokens_word_lyrics))
print("\nSubword-level Tokenizer:")
print(" | ".join(tokens_subw_lyrics))


Lyrics:
โอ๊ย อันตั้งบะเล่อ เทิ่งบ้าง
ใจคงใจคอไม่ดีเลยครับ
นี่คนหรือว่านางฟ้า
ดูเธอในติกตอก กดใจไม่พัก
เห็นเธอนะโพสต์ไลก์ทัก
ก็เลยกดหัวใจ จัดไปเลยครับ
แล้วเธอก็ตอบอย่างไว
ทักมาทันใจว่าสวัสดีค่ะ
แปลกๆเด้อเหมือนเธอจะมีสูตร
เธอก็แปะลิงก์ให้ผมเข้าไปดู
ระบบปลอดภัยฝากถอนโอนไวว่าแล้วกู
นึกว่ามีใจที่ไหนได้เธอมีสูตร
สองหนึ่งสอง ละก็สองสองสี่ ละก็สองสามหก
ละก็สองสี่แปด
นึกว่าเธอมีใจที่ไหนกันได้เธอมี
สูตรคูณแม่แปด
แปดหนึ่งแปด แปดสองสิบหก แปดสามยิบสี่
แปดสี่สามแปด
บ่แม่นๆ สามสอง บ่แม่นสามแปด
ที่หนึ่งระบบหวย ไป ไป ไป ฟ้าว ฟ้าว
ระบบหนูก็ปลอดภัย โอ้ย เซา เซา เซา
ฝากถอนหนูก็โอนไว โอ้ย หยังอีก หยังอีก
ขอให้พี่เชื่อใจ (โอ๊ยบักปอบ)
สองหนึ่งสอง ละก็สองสองสี่ สองสามหก
ละก็สองสี่แปด
ถ้ายอดหนูไม่ได้
พี่ช่วยได้ไหม
สักสองร้อยก็ได้
หนูอยากสิไห้
บ่เล่น บ่เล่น มาชวนเฮ็ดหยัง ปวดหัวแหน่สู
ฟ้าว ฟ้าว ฟ้าว ฟ้าว
แปลกๆเด้อเหมือนเธอจะมีสูตร
เธอก็แปะลิงก์ให้ผมเข้าไปดู
ระบบปลอดภัยฝากถอนโอนไวว่าแล้วกู
นึกว่ามีใจที่ไหนได้เธอมีสูตร
เธอขาว เธอสวย เธอเซ็กซี่ เธอลงสตอรี่มีเว็บให้ดู
ทำไมเธอถึงทำแบบนี้ กดใจให้ทุกทีอย่างเศร้าเลยตรู


เมื่อคำนึงถึงบริบทของ การใช้ subword-level tokenization จะให้ผลลัพธ์ที่ดีกว่าในบริบทของเนื้อเพลงอักษรไทยที่มีการใช้สแลง ภาษาถิ่น และสำนวนนอกเหนือจากภาษาไทยมาตรฐาน ซึ่งมีความเสี่ยงต่อการเกิดปัญหา OOV สูง **ดังนั้น จึงตัดสินใจเลือกใช้ subword-level tokenization** ทั้งนี้ ต้องแลกกับข้อความอักษรอื่น ๆ ที่ไม่ได้รับการแบ่งเป็น subword ได้อย่างเหมาะสมตามหลักภาษานั้น ๆ ด้วย

#### Bonus?

In [9]:
# Bonus: ร รักห่วยแตก - เอครู (ECTRU)

lyrics = clean_text(corpus[508])

# Word-level
tokens_word_lyrics = pythainlp.tokenize.word_tokenize(lyrics)
tokens_word_lyrics = [t.strip() for t in tokens_word_lyrics if t.strip()]

# Subword-level
tokens_subw_lyrics = pythainlp.tokenize.subword_tokenize(lyrics)
tokens_subw_lyrics = [t.strip() for t in tokens_subw_lyrics if t.strip()]

print("Lyrics:")
print(lyrics)
print("\nWord-level Tokenizer:")
print(" | ".join(tokens_word_lyrics))
print("\nSubword-level Tokenizer:")
print(" | ".join(tokens_subw_lyrics))

Lyrics:
ก เอ๋ย ก กู
ข เขา อยู่บนหัว
ค ควาย คือกู
น นั่ง ง โง่ อยู่ตรงนี้
จ ใจ แทบขาด
ช ช้ำเจ็บปวดร้าว
ม เมา โดดเดี่ยวเดียวดาย
ต แตกสลาย เพียง ล ลำพัง
ม มึง ท ทิ้ง ก กูไป
ศ เศร้าเสียน้ำตา จ ใจกูสลาย
ม มึง ไปกับ ม มัน
ท ทิ้ง ก กู คนนี้ ป ไปไกลแสนไกล
ร รัก ของกู มันห่วย ต ต ต แตก
ฝ ฝันสลาย เมื่อ ธ เธอเดินจากไป
ซ ซึมเศร้า จ เจ็บเพียงลำพัง
ก กู ล ล้มกี่ครั้ง ม ไม่หวั่นไหว
ต แต่ง พ เพลงนี้ ฮ ฮีลใจ ก กูเอง
จ ใจ แทบขาด
ช ช้ำเจ็บปวดร้าว
ม เมา โดดเดี่ยวเดียวดาย
ต แตกสลาย เพียง ล ลำพัง
ม มึง ท ทิ้ง ก กูไป
ศ เศร้าเสียน้ำตา จ ใจกูสลาย
ม มึง ไปกับ ม มัน
ท ทิ้ง ก กู คนนี้ ป ไปไกลแสนไกล
ร รัก ของกู มันห่วย ตอ แอ กอ แตก
ม มึง ท ทิ้ง ก กูไป
ศ เศร้าเสียน้ำตา จ ใจกูสลาย
ม มึง ไปกับ ม มัน
ท ทิ้ง ก กู คนนี้ ป ไปไกลแสนไกล
ร รัก ของกู มันห่วย ตอ แอ กอ แตก
ก กู อยู่ได้ แม้ไม่มี ม มึง
ก กู อยู่ได้ ไม่ต้องมี ม มึง
ก กู คนนี้ จะดูแล ต ตัวเอง
ตอ แอ กอ แตก

Word-level Tokenizer:
ก | เอ๋ย | ก | กู | ข | เขา | อยู่ | บน | หัว | ค | ควาย | คือ | กู | น | นั่ง | ง | โง่ | อยู่ | ตรงนี้ | จ | ใจ | แทบ | ขาด | ช | ช้ำ | 

In [10]:
# Bonus: หีบหาย - เต็ด ยุทธนา บุญอ้อม (DJ Pated)

lyrics = clean_text(corpus[814])

# Word-level
tokens_word_lyrics = pythainlp.tokenize.word_tokenize(lyrics)
tokens_word_lyrics = [t.strip() for t in tokens_word_lyrics if t.strip()]

# Subword-level
tokens_subw_lyrics = pythainlp.tokenize.subword_tokenize(lyrics)
tokens_subw_lyrics = [t.strip() for t in tokens_subw_lyrics if t.strip()]

print("Lyrics:")
print(lyrics)
print("\nWord-level Tokenizer:")
print(" | ".join(tokens_word_lyrics))
print("\nSubword-level Tokenizer:")
print(" | ".join(tokens_subw_lyrics))


Lyrics:
หีบมากมายหลายหีบ ยกหีบหนี
หีบมากมีหนีหีบ หนีบหนีหาย
เห็นหนีหีบ หนีบหนีกันมากมาย
เห็น
หีบหาย
หลายหีบ ใครหนีบ หนี
อย่ามาแหวง อย่ามาแหวง
อย่ามาแหวง อย่ามาหนี
ไอ้ชิบหาย เอ้าหายไปไหนล่ะหีบ
เอ้าหีบหาย ไอ้ชิบหายใครหนีบหนี
จะหนีบหนี หนีหีบ หีบสี หีบสวย
พี่แหวงช่วยด้วย หีบหนูโดนหนีบหนี
หีบมากมายหลายหีบ ยกหีบหนี
หีบมากมีหนีหีบ หนีบหนีหาย เห็นหนีหีบ
หนีบหนีกันมากมาย เห็นหีบหายหลายหีบ
ใครหนีบ
อย่ามาแหวง อย่ามาแหวง
อย่ามาแหวง อย่ามาหนี
ไอ้ชิบหาย เอ้าหายไปไหนล่ะหีบ
เอ้าหีบหาย ไอ้ชิบหายใครหนีบหนี
จะหนีบหนี หนีหีบ หีบสี หีบสวย
พี่แหวงช่วยด้วย หีบหนูโดนหนีบหนี
อย่ามาแหวงหีบหนี
อย่ามาแหวงหนีหีบ
อย่ามแหวงหนีบหนี
อย่ามาแหวงหีบหนู
ไอ้ชิบหาย เอ้าหายไปไหนล่ะหีบ
เอ้าหีบหาย ไอ้ชิบหายใครหนีบหนี
จะหนีบหนี หนีหีบ หีบสี หีบสวย
พี่แหวงช่วยด้วย หีบหนูโดนหนีบหนี
ไอ้ชิบหาย หาย หาย หาย หาย
เอ้าหีบหาย จะหนีบหนี หนี หนี หนี หนี
พี่แหวงช่วยด้วย หีบหนูโดนหนีบหนี

Word-level Tokenizer:
หีบ | มากมาย | หลาย | หีบ | ยก | หีบ | หนี | หีบ | มาก | มี | หนี | หีบ | หนีบ | หนีหาย | เห็น | หนี | หีบ | หนีบ | หนี | กัน | มากม

### Tokenize (final)


นำ `corpus` มา tokenize ในระดับ subword-level

In [11]:
def tokenize(text):
    tok = pythainlp.tokenize.word_tokenize(clean_text(text))
    tok = [t.strip() for t in tok if t.strip()]
    return tok


In [12]:
tokens = []
for doc in corpus:        
    tokens.extend(tokenize(doc))

token_dist = FreqDist(tokens)
print(token_dist)
print(token_dist.most_common(100))


<FreqDist with 10518 samples and 265764 outcomes>
[('เธอ', 8913), ('ไม่', 6509), ('ฉัน', 5530), ('จะ', 4434), ('ที่', 4401), ('ให้', 3932), ('ก็', 3555), ('ได้', 3553), ('มี', 3403), ('ไป', 3357), ('มัน', 3108), ('มา', 2707), ('ว่า', 2654), ('เป็น', 2464), ('รัก', 2104), ('แค่', 2097), ('แต่', 2086), ('คน', 2063), ('อยู่', 2012), ('ยัง', 1765), ('บ่', 1761), ('ใน', 1725), ('อยาก', 1712), ('ใจ', 1664), ('ต้อง', 1549), ('กัน', 1467), ('แล้ว', 1440), ('ของ', 1421), ('เรา', 1395), ('ใคร', 1379), ('you', 1362), ('กับ', 1207), ('เคย', 1196), ('รู้', 1177), ('เขา', 1111), ('ถ้า', 1102), ('อ้าย', 1084), ('i', 1073), ('หัวใจ', 1027), ('เลย', 980), ('และ', 957), ('นั้น', 904), ('คือ', 897), ('ไหม', 891), ('นี้', 884), ('สิ', 883), ('บอก', 837), ('เจ้า', 837), ('อย่า', 807), ('คง', 778), ('ดี', 735), ('คิด', 701), ('กะ', 692), (')', 684), ('ฮัก', 666), ('ๆ', 661), ('ขอ', 660), ('ทำ', 657), ('เพราะ', 655), ('ไว้', 620), ('เจอ', 610), ('วัน', 587), ('ไหน', 582), ('น้อง', 570), ('อะไร', 555), ('ถึง'

********Information Extraction & Named Entity Recognition********

เช็คให้แน่ใจว่า Corpus มีลักษณะเป็น list ของเนื้อเพลงี่ต้องการหรือไม่เช็คจาก index[]

In [13]:
print(corpus[0])

ยังมีเรื่องที่อยากทำอยู่เต็มกระดาน
เริ่มจากทำเพลงแล้วให้เพลงมันทำงาน
แล้วปัญหาของมึงก็เรื่องของมึงดิ
ถ้าเกาะผมมันไม่ดังคุณก็ทำกันเองดิ
Get out บอกให้พวกมึงเดินหลีก
เลือกจะเดินตรงแล้วปล่อยให้พวกมึงเดินดีด
จุดไฟนำทางก็เพียงแค่เผามัน
Oh everyday hit the ควัน
Get hype up with my beat
ดีดผมดีดเช้าดีดเย็น
แค่เล็งให้ลงหลุมเรื่องลูกแก้วผมดีดเป็น
มีความคิดจริตเย็น
For the next step ให้กับชีวิตต้องสิบเต็ม
ไม่รู้จะพูดยังไง Just wanna kiss her
ขาดเธอไม่ได้ก็ใจมัน Need ya
And I'd love to know what's going on in your mind
You and me We decied
When I wasn't drunk Crazy or high
แล้วมันต้องมีสักวันและก่อนที่มันจะสาย
จงเชื่อในสิ่งที่ทำให้มันมีความหมาย
เหนื่อยไหมคนดี
ที่เธอมีพี่เป็นแฟน
แก้วแหวนผมเคยมีแต่เอาไปจำนำทำนมให้เธอแทน
พอมีคนที่รักพี่ทุ่มให้ทั้งใจ
แล้วเธอกลายเป็นคนที่สวยแต่รูปไปได้ไง
ความสวยงามมันต้องงามจากข้างใน
แบบเธอที่ฉันมองข้ามไปเพิ่งจะรู้ตัวและเข้าใจ
ชอบเธอนั้นมันก็คือเรื่องนึง
แต่ถ้าเธอไปชอบเขานั่นมันก็เรื่องของมึง
ก็ทรัพย์สลึงที่ฉันมีมันเจือจาง
คงไม่มั่นคงจนเรื่องรักเราเลือนลาง
ชอบตอนเมินท

***สร้าง vector แบบ TF_IDF และในไปสร้างตารางข้อมูลแบบ (จำนวนเพลง x จำนวนคำทั้งหมด)***

In [ ]:
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,
    max_df = 0.95,
    min_df = 2
    )
tfidf_matrix = tfidf_vec.fit_transform(corpus)
print(tfidf_matrix.shape)

d:\AITwork\NLP\term_pro\nlp_pro\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


(880, 5341)


**สร้างNMFโดยที่พยายามจะควบคุม n_components ให้เหมาะสมกับข้อมูล**

In [74]:
nmf_model = NMF(
    n_components=7,
    random_state=42
    )

w = nmf_model.fit_transform(tfidf_matrix)

print(w.shape) # W_Matrix
print(w[0: 3]) # แต่ละเพลงเกี่ยวข้องกับ topic นั้นๆอย่างไร
print(nmf_model.components_.shape) # H_Matrix
print(nmf_model.components_[0: 3]) # topic นั้นๆเกี่ยวข้องกับคำไหน

(880, 7)
[[0.07885142 0.00016989 0.13168004 0.03257712 0.02866434 0.05249694
  0.10202147]
 [0.         0.00974871 0.         0.         0.00752357 0.
  0.0824124 ]
 [0.04052188 0.02686425 0.         0.         0.         0.02189471
  0.08003624]]
(7, 5341)
[[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 4.68206713e-05
  7.99303925e-04 3.39437789e-02]
 [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 5.07619217e-04
  0.00000000e+00 3.24630280e-02]
 [5.99274672e-02 1.27099304e-02 5.34092865e-02 ... 0.00000000e+00
  0.00000000e+00 7.90817610e-03]]


d:\AITwork\NLP\term_pro\nlp_pro\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [75]:
#พยายามจะดูว่าใน tfidf ของแต่ละ tokenizer มีคำอะไรบ้าง
vocab = tfidf_vec.get_feature_names_out()
print(vocab[5000])

แว่น


In [76]:
#พยายามจะดึงเอาคำที่เด็ดๆจาก nmf_model
top_words = nmf_model.components_.argsort()[-1: -11: -1]
print(top_words)

[[ 594 1915   16 ... 4950 2554 2731]
 [  31    1    2 ... 5288 2912  958]
 [   0   15   14 ... 1418 4560 2110]
 ...
 [5273 5274 5277 ...  427  350  775]
 [5330    1    2 ... 4221 4083 2397]
 [5327 3670 5313 ... 1418 1503 4364]]


สังเกตุว่าจากโค้ดข้างบน เราพยายามที่จะเอา10คำที่ RELATE กับ Topics มากที่สุด แต่ด้วยความที่มันเป็น 2มิติ เราจะได้ผลลัพธ์ออกมาเป็นลิสต์ลำดับจากทุกๆแถว

In [77]:
#เราต้องพยายามอ่านทีละแถวข คือ ทีละ topic
for i, topic in enumerate(nmf_model.components_):
    #เราพยายามเอาคำที่เด็ดที่สุด10คำ โดยเรียงด้วย argsort แล้วเราเอา 10 คำท้ายที่มีค่ามากสุด
    topten = topic.argsort()[-1: -11: -1]

    top_words = []
    for index in topten:
        word = vocab[index]
        top_words.append(word)

print(top_words)    

['พี่', 'ผม', 'แม่', 'น้อง', 'มา', 'ไม่', 'จะ', 'ให้', 'ละ', 'ก็']


เราจะสร้างเครื่องทือที่แอบส่องดูว่าหัวข้อที่โมเดลพยายามจะเรียนรู้คืออะไร

In [78]:
# Helper function to print topics
def print_topics(model, vectorizer, n_top_words=3):
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx}: "
        message += " ".join([feature_names[i]
                            for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

***สังเกตุว่ามีการเชื่อมโยงของคำที่โมเดลพยายามจะจับมาอยู่ด้วยกัน***
3 ใน 7 มีการใช้สรรพนามบุคคลที่ 1 ร่วมกับ สรรพนามบุคคลที่ 2 แม้จะเป็นภาษาคนละระดับ หรือคนละภาษาก็ตาม
ใน topic ที่ 6 มีการเชื่อมโยงถึงคนในครอบครัว

In [ ]:
print_topics(nmf_model, tfidf_vec)

--- NMF Topics ---
Topic #0: เธอ ฉัน จะ
Topic #1: บ่ อ้าย เจ้า
Topic #2: you i me
Topic #3: ไม่ ฉัน มัน
Topic #4: ที่ เรา จะ
Topic #5: กู มึง ไม่
Topic #6: พี่ ผม แม่


เนื่องจากที่ผ่านมาเป็นการสกัดแบบ Unsupervised Learning เราจะไปต่อกันที่แบบ Supervised Learning ในการทำ ***Named Entity Recognition***

เราจะลองใช้ THAI NER 2.2 (Named Entity Recognition) จาก https://github.com/wannaphong/thai-ner

In [81]:
from datasets import load_dataset
from pythainlp.tokenize import word_tokenize, subword_tokenize

d:\AITwork\NLP\term_pro\nlp_pro\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


***เปรียบเทียบว่า Tokenizer แบบไหน ตัดคำได้ตรงกับ "ขอบเขตคำนามเฉพาะ" (Entity Boundary) ใน Thai NER 2.2 มากที่สุด***

In [83]:
# 1. โหลดข้อมูล Thai NER 2.2
ner_dataset = load_dataset("pythainlp/thainer-corpus-v2.2")
sample_ner = ner_dataset["train"][0]
# 2. เอาคำศัพท์ที่ตัดมาแล้วมารวมเป็นประโยคยาวๆ ติดกัน
original_words = sample_ner["words"]
text_full = "".join(original_words)
print("ออริจินอล text:", text_full)
# 3. ลองใช้ Tokenizer ระดับดั้งเดิม (Word-level)
tokens_word = word_tokenize(text_full)
print("\n[Word-level]:", " | ".join(tokens_word))
# 4. ลองใช้ Tokenizer ระดับย่อย (Subword-level)
tokens_subw = subword_tokenize(text_full)
print("\n[Subword-level]:", " | ".join(tokens_subw))
# 5. เปรียบเทียบความยาวของ list ว่า tokenize ออกมาได้กี่เรคคอร์ด เทียบกับ original
print("\n--- สรุปจำนวน Token ---")
print(f"Original (จาก NER dataset): {len(original_words)} คำ")
print(f"Word-level: {len(tokens_word)} คำ")
print(f"Subword-level: {len(tokens_subw)} คำ")

ออริจินอล text: ทักษิณ ชินวัตร

[Word-level]: ทักษิณ |   | ชินวัตร

[Subword-level]: ทั | ก | ษิ | ณ |   | ชิ | น | วั | ต | ร

--- สรุปจำนวน Token ---
Original (จาก NER dataset): 3 คำ
Word-level: 3 คำ
Subword-level: 10 คำ


***ใช้ Tokenizer มาสกัดคำ (Extract) และดูเปอร์เซ็นต์คำคล้าย Entity***

In [84]:
# หารายชื่อคำที่เป็น Entity จริงๆ จาก Thai NER (สมมติเก็บมาสัก 1,000 ประโยคแรก)
important_entities = set()
for i in range(1000):
    words = ner_dataset["train"][i]["words"]
    tags = ner_dataset["train"][i]["ner"]
    for w, t in zip(words, tags):
        if t != 2: # 2 คือ 'O' (ไม่ใช่ Entity)
            important_entities.add(w.lower())
            
print(f"หาคำนามเฉพาะ (ชื่อคน, สถานที่ ฯลฯ) เจอทั้งหมด {len(important_entities)} คำ")

# นำ vocab (คำศัพท์) จาก TF-IDF ของคุณมาเช็คว่ามีคำนามเฉพาะหลุดมาอยู่ในเนื้อเพลงกี่คำ
vocab_set = set(vocab)
overlap = vocab_set.intersection(important_entities)

print(f"\nจากคำศัพท์ในเนื้อเพลงทั้งหมด {len(vocab)} คำ มีคำนามเฉพาะปนอยู่ {len(overlap)} คำ")
# ลองสุ่มปรินท์ดู
print("ตัวอย่างคำนามเฉพาะในเนื้อเพลง:", list(overlap)[:20])


หาคำนามเฉพาะ (ชื่อคน, สถานที่ ฯลฯ) เจอทั้งหมด 2148 คำ

จากคำศัพท์ในเนื้อเพลงทั้งหมด 5341 คำ มีคำนามเฉพาะปนอยู่ 515 คำ
ตัวอย่างคำนามเฉพาะในเนื้อเพลง: ['จ', 'ดา', 'ใหญ่', 'แห่ง', 'สตาร์', 'เอ', 'สา', 'แปด', 'แทนคุณ', 'เมือง', 'ท', 'สำ', '7', 'คุ้มครอง', 'ศาล', 'หลัง', 'ช่วง', 'ประเสริฐ', 'ฟี', 'เสก']
